# OIBSIP Data Analytics — Level 2, Task 3: Fraud Detection
## Machine Learning Pipeline for Imbalanced Financial Transactions

**Objective:** detect potentially fraudulent transactions while addressing class imbalance.

> **Data note:** `transactions_demo.csv` is a reproducible synthetic dataset created for this portfolio demonstration. It is not real customer or bank data and must not be presented as actual fraud statistics. For a real-world analysis, replace it with the Kaggle Credit Card Fraud Detection dataset and adapt the feature columns.

## 1. Problem statement
Fraud is rare compared with legitimate transactions. A model that predicts every transaction as legitimate may show very high accuracy while detecting no fraud. This project therefore focuses on **precision, recall, F1-score and ROC-AUC**, alongside the confusion matrix.

- **Precision:** among transactions flagged as fraud, how many truly are fraud?
- **Recall:** among all fraud transactions, how many did the model catch?
- **F1:** harmonic balance between precision and recall.
- **ROC-AUC:** how well scores rank fraud above legitimate transactions across thresholds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve)
from pathlib import Path

DATA_PATH = Path("transactions_demo.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dataset inspection and cleaning

In [ ]:
print("Shape:", df.shape)
display(df.dtypes)
display(df.isna().sum().to_frame("Missing values"))
display(df.describe(include="all").T)

# Remove duplicate transaction IDs and repair missing / invalid fields
df = df.drop_duplicates(subset="transaction_id").copy()
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["amount"] = df["amount"].fillna(df["amount"].median())
df["hour"] = pd.to_numeric(df["hour"], errors="coerce").fillna(df["hour"].median())
df["international"] = df["international"].fillna(0).astype(int)
df["channel"] = df["channel"].fillna("Unknown")
df["merchant_category"] = df["merchant_category"].fillna("Unknown")
df["fraud"] = df["fraud"].astype(int)
print("Cleaned shape:", df.shape)

## 3. Class imbalance analysis

In [ ]:
class_counts = df["fraud"].value_counts().sort_index()
class_pct = df["fraud"].value_counts(normalize=True).sort_index() * 100
display(pd.DataFrame({"count": class_counts, "percent": class_pct}))
plt.figure(figsize=(7,4))
plt.bar(["Legitimate (0)", "Fraud (1)"], class_counts.values)
plt.title("Transaction Class Distribution"); plt.ylabel("Number of transactions")
for i,v in enumerate(class_counts.values): plt.text(i,v+class_counts.max()*0.01,f"{v:,}",ha="center")
plt.tight_layout(); plt.show()
print(f"Fraud rate: {df.fraud.mean():.3%}")

**Interpretation:** Fraud is the minority class. Accuracy alone can conceal missed fraud cases; model evaluation must examine recall and precision and the number of false alarms.

## 4. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x="amount", hue="fraud", bins=45, stat="density", common_norm=False, element="step")
plt.title("Transaction Amount Distribution: Fraud vs Legitimate"); plt.tight_layout(); plt.show()

hourly = df.groupby("hour", as_index=False)["fraud"].mean()
plt.figure(figsize=(8,4))
sns.lineplot(data=hourly, x="hour", y="fraud", marker="o")
plt.title("Fraud Rate by Hour"); plt.xlabel("Hour of day"); plt.ylabel("Fraud rate")
plt.xticks(range(0,24,2)); plt.tight_layout(); plt.show()

display(df.groupby("fraud")["amount"].agg(["count","mean","median","std"]))

**Observations to report:** compare the amount distributions and hourly fraud rates. Peaks in this synthetic dataset are generated by the simulation rules, so they are demonstrations—not evidence about real banking behaviour.

## 5. Train/test split and preprocessing

In [ ]:
features = ["amount", "hour", "channel", "international", "merchant_category"]
X = df[features]
y = df["fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
numeric_features = ["amount", "hour", "international"]
categorical_features = ["channel", "merchant_category"]
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Fraud rate train/test:", y_train.mean(), y_test.mean())

## 6. Model training

In [ ]:
models = {
    "Logistic Regression": Pipeline([("prep", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))]),
    "Random Forest": Pipeline([("prep", preprocessor),
        ("model", RandomForestClassifier(n_estimators=180, class_weight="balanced",
                                        min_samples_leaf=2, random_state=42))])
}
results, predictions, probabilities = {}, {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype(int)  # threshold can be tuned for business needs
    predictions[name], probabilities[name] = pred, prob
    results[name] = {
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    }
display(pd.DataFrame(results).T.round(4))

## 7. Evaluation and charts

In [ ]:
for name in models:
    print("\\n", name)
    print(classification_report(y_test, predictions[name], zero_division=0))
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Legitimate","Fraud"], yticklabels=["Legitimate","Fraud"])
    plt.title(f"Confusion Matrix — {name}"); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.show()

plt.figure(figsize=(7,5))
for name, prob in probabilities.items():
    fpr,tpr,_ = roc_curve(y_test, prob)
    plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,prob):.3f})")
plt.plot([0,1],[0,1],"--",label="Random baseline")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curves")
plt.legend(); plt.tight_layout(); plt.show()

metric_df = pd.DataFrame(results).T
metric_df[["Precision","Recall","F1","ROC-AUC"]].plot(kind="bar",figsize=(8,5),ylim=(0,1))
plt.title("Model Comparison"); plt.ylabel("Score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 8. Metric interpretation and threshold trade-off
- **Recall is often prioritized** when missing a fraudulent transaction is costly.
- **Precision matters** because too many false alerts create customer friction and investigation workload.
- A lower classification threshold generally catches more fraud (higher recall) but can increase false positives (lower precision).
- Select the operating threshold using business costs, validation data, and review capacity—not only the default 0.50 threshold.

## 9. Feature importance / coefficient analysis
For Random Forest, inspect feature importances after preprocessing. This is an association/importance view, not proof of causality.

In [ ]:
rf = models["Random Forest"]
feature_names = rf.named_steps["prep"].get_feature_names_out()
importance = rf.named_steps["model"].feature_importances_
imp = pd.Series(importance,index=feature_names).sort_values(ascending=False).head(12)
display(imp.to_frame("importance"))
imp.sort_values().plot(kind="barh",figsize=(8,5)); plt.title("Top Random Forest Feature Importances")
plt.tight_layout(); plt.show()

## 10. Scalability discussion (1 million transactions/hour)
One million/hour is about **278 transactions/second** on average. A production design could:
1. Stream transactions through Kafka or a managed event queue.
2. Run a lightweight feature service and batch/online inference workers.
3. Scale workers horizontally and monitor latency, throughput, and queue lag.
4. Use a model registry and versioned deployment; support rollback.
5. Monitor precision/recall drift, data drift, false-positive rates, and delayed fraud labels.
6. Keep sensitive data encrypted, access-controlled, minimized, and audited.
7. Route high-risk events for step-up verification or human review rather than automatically blocking every alert.

## 11. Conclusion
This project demonstrates an imbalanced classification workflow: cleaning, exploratory analysis, stratified split, class-weighted model training, and evaluation using precision, recall, F1 and ROC-AUC. Compare models using the business cost of missed fraud versus false alarms. **Replace the demo data with the official dataset before making real-world claims.**